In [6]:
from sentence_transformers import SentenceTransformer

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def get_embedding(text,input_type="document"):

    return embedding_model.encode(text).tolist()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9271.67it/s]


In [7]:
embed=get_embedding("RAG with MongoDB")
len(embed)

384

In [8]:
from langchain_community.document_loaders import PyPDFLoader




C:\Users\karth\AppData\Local\Temp\ipykernel_12552\1181437673.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [9]:
loader= PyPDFLoader("https://investors.mongodb.com/node/12236/pdf")
data=loader.load()
type(data)
 
 

list

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(data)
chunks



[Document(metadata={'producer': 'West Corporation using ABCpdf', 'creator': 'PyPDF', 'creationdate': '2026-06-06T03:30:11+00:00', 'title': 'MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results', 'source': 'https://investors.mongodb.com/node/12236/pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results\nMay 30, 2024\nFirst Quarter Fiscal 2025 Total Revenue of $450.6 million, up 22% Year-over-Year\nContinued Strong Customer Growth with Over 49,200 Customers as of April 30, 2024\nMongoDB Atlas Revenue up 32% Year-over-Year; 70% of Total Q1 Revenue\xa0\nNEW YORK, May 30, 2024 /PRNewswire/ -- MongoDB, Inc. (NASDAQ: MDB) today announced its financial results for the first quarter ended April 30,\n2024.\n\xa0\n  \xa0\n"MongoDB\'s delivered solid first quarter results, highlighted by 32% Atlas revenue growth. At the same time, we had a slower than expected start to\nthe year for both Atlas consumpti

In [11]:
docs_to_insert=[
    {
        "text":doc.page_content,
        "embedding":get_embedding(doc.page_content)
    } for doc in chunks
]

In [ ]:
from pymongo import MongoClient

client = MongoClient("")
collection=client["sample_mflix"]["ragpdf"]

result=collection.insert_many(docs_to_insert)
result

InsertManyResult([ObjectId('6a3e67ce9fd3831990ab24b6'), ObjectId('6a3e67ce9fd3831990ab24b7'), ObjectId('6a3e67ce9fd3831990ab24b8'), ObjectId('6a3e67ce9fd3831990ab24b9'), ObjectId('6a3e67ce9fd3831990ab24ba'), ObjectId('6a3e67ce9fd3831990ab24bb'), ObjectId('6a3e67ce9fd3831990ab24bc'), ObjectId('6a3e67ce9fd3831990ab24bd'), ObjectId('6a3e67ce9fd3831990ab24be'), ObjectId('6a3e67ce9fd3831990ab24bf'), ObjectId('6a3e67ce9fd3831990ab24c0'), ObjectId('6a3e67ce9fd3831990ab24c1'), ObjectId('6a3e67ce9fd3831990ab24c2'), ObjectId('6a3e67ce9fd3831990ab24c3'), ObjectId('6a3e67ce9fd3831990ab24c4'), ObjectId('6a3e67ce9fd3831990ab24c5'), ObjectId('6a3e67ce9fd3831990ab24c6'), ObjectId('6a3e67ce9fd3831990ab24c7'), ObjectId('6a3e67ce9fd3831990ab24c8'), ObjectId('6a3e67ce9fd3831990ab24c9'), ObjectId('6a3e67ce9fd3831990ab24ca'), ObjectId('6a3e67ce9fd3831990ab24cb'), ObjectId('6a3e67ce9fd3831990ab24cc'), ObjectId('6a3e67ce9fd3831990ab24cd'), ObjectId('6a3e67ce9fd3831990ab24ce'), ObjectId('6a3e67ce9fd3831990ab24

In [13]:
from pymongo.operations import SearchIndexModel
import time

index_name="vector_index"
search_index_model=SearchIndexModel(
    definition={
        "fields":[
            {
                "type":"vector",
                "numDimensions":384,
                "path":"embedding",
                "similarity":"cosine"
            }
        ]
    },
    name=index_name,
    type="vectorSearch"


)
collection.create_search_index(model=search_index_model)

'vector_index'

In [14]:
query_embeddings=get_embedding("AI Technology")

In [15]:
results=collection.ragpdf.aggregate([
    {
        "$vectorSearch":{
                "index":"vector_index",
                "queryVector":query_embeddings,
                "path":"embedding",
                "numCandidates":384,
                "limit":5
        }

    }
]

)

In [16]:
results

In [17]:
array_of_results=[]
for doc in results:
    array_of_results.append(doc)
array_of_results

[]

In [18]:
def get_query_result(query):
    query_embedding=get_embedding(query, input_type="query")
    print(query_embedding)
    pipeline = [
        {
            "$vectorSearch":{
                "index":"vector_index",
                "queryVector":query_embedding,
                "path":"embedding",
                "numCandidates":384,
                "limit":5
            
            }
        },{
            "$project":{
                "_id":0,
                "text":1
            }
        }
    ]
    results=collection.aggregate(pipeline)
    print(results)
    array_of_results=[]
    for doc in results:
        array_of_results.append(doc)
    return array_of_results


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="",
    temperature=0
)
question= input("Ask your Doubts: ")
results=get_query_result(question)
context = "\n\n".join([doc["text"] for doc in results])



prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

response = llm.invoke(prompt)
print("YOUR DOUBT: ", question)




print(response.content)


[-0.01186967920511961, 0.037047721445560455, 0.0408061183989048, 0.029772989451885223, 0.010610838420689106, 0.010564274154603481, 0.06946512311697006, -0.00814379844814539, -0.007282191421836615, 0.004363973159343004, 0.007767447270452976, -0.01280568353831768, 0.043397970497608185, -0.050401996821165085, 0.057747237384319305, -0.003372742095962167, 0.05318738520145416, 0.06233726441860199, -0.002025144873186946, 0.004193440545350313, -0.02714400738477707, 0.005583900958299637, -0.05347176268696785, 0.0592176727950573, -0.03789053484797478, 0.04907294735312462, 0.013286907225847244, -0.023327134549617767, 0.0696476548910141, 0.023512914776802063, -0.053727541118860245, 0.04712279513478279, -0.07791165262460709, 0.08425358682870865, -0.017901474609971046, 0.08260532468557358, -0.020495383068919182, 0.02953711338341236, -0.07176327705383301, -0.020924905315041542, -0.06612340360879898, -0.02316083014011383, 0.009655647911131382, -0.00721373874694109, 0.041347332298755646, 0.025215908885